# 05 — Final Validation, Calibration, Lift and Reporting

Clean rerun directly from the 40,000-row dataset.

There are no dependencies on repository `data/`, `results/`, or `figures/` folders, no cached CSV/JSON/NPY inputs, and no generated-file writes.

The notebook reconstructs the eligible expanding-window predictions itself using the fresh tuned, unweighted pre-call parameters from Notebook 02. `duration` is excluded throughout.

## 1. Setup and reconstruction of eligible forward predictions

**Coding step.** Reconstruct eligible forward predictions directly from the dataset and transferred tuned pre-call settings, without cached result files.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

warnings.filterwarnings("ignore")
SEED = 42
DATA_SOURCE = "term-deposit-marketing-2020-labelled.csv"

NUM = ["age","balance","day","campaign"]
CAT = ["job","marital","education","default","housing","loan","contact","month"]
PRE = NUM + CAT

TUNED = {
    "LR": {"C":8.483428982440726, "penalty":"l2", "class_weight":None},
    "HGB": {"max_iter":200, "max_depth":8, "learning_rate":0.05, "max_leaf_nodes":15, "l2_regularization":0.0, "min_samples_leaf":20, "class_weight":None, "early_stopping":True}
}

df = pd.read_csv(DATA_SOURCE)
if "y_binary" in df.columns: df = df.drop(columns="y_binary")
df["y_binary"] = df["y"].eq("yes").astype(int)
y = df["y_binary"].to_numpy()
assert "duration" not in PRE

def prep(): return ColumnTransformer([("num",StandardScaler(),NUM),("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),CAT)])
def pipe(kind):
    model=LogisticRegression(solver="liblinear",max_iter=3000,random_state=SEED,**TUNED["LR"]) if kind=="LR" else HistGradientBoostingClassifier(random_state=SEED,**TUNED["HGB"])
    return Pipeline([("prep",prep()),("model",model)])

ordered=df.copy(); ordered["row_order"]=np.arange(len(ordered)); ordered["period_id"]=ordered["month"].astype(str).ne(ordered["month"].astype(str).shift()).cumsum()
gate=ordered.groupby("period_id").agg(month=("month","first"),start_row=("row_order","min"),test_n=("y_binary","size"),positives=("y_binary","sum")).reset_index()
gate["train_n"]=gate["start_row"]; gate["negatives"]=gate["test_n"]-gate["positives"]; gate["period_label"]=[f"P{int(p):02d}-{m}" for p,m in zip(gate.period_id,gate.month)]
gate["eligible"]=gate["train_n"].ge(5000)&gate["positives"].ge(100)&gate["negatives"].ge(100)
parts=[]
for _,row in gate.loc[gate["eligible"]].iterrows():
    pid=int(row["period_id"]); tr=ordered.index[ordered["period_id"]<pid].to_numpy(); te=ordered.index[ordered["period_id"]==pid].to_numpy()
    for kind in ["LR","HGB"]:
        est=pipe(kind); est.fit(df.loc[tr,PRE],y[tr]); p=est.predict_proba(df.loc[te,PRE])[:,1]; parts.append(pd.DataFrame({"model":kind,"period_id":pid,"period_label":row["period_label"],"y_true":y[te],"score":p}))
forward=pd.concat(parts,ignore_index=True)
print("Eligible periods:",gate.loc[gate.eligible,"period_id"].astype(int).tolist()); print("Forward rows per model:",forward.groupby("model").size().to_dict()); print("Forward positives per model:",forward.groupby("model")["y_true"].sum().to_dict()); print("Duration used:","duration" in PRE)

## 2. Period-local lift by call depth

Customers are ranked **within each eligible forward block**, not globally across different periods. For each block and call depth, the top `ceil(depth × block size)` customers are selected. Aggregate lift is calculated relative to random calling at the same call volume. Confidence intervals use deterministic period-level bootstrap resampling.

**Coding step.** Compute period-local lift, captured subscribers and random expectation at 5%, 10%, 20% and 50% call depth.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
DEPTHS=[0.05,0.10,0.20,0.50]
def period_depth_stats(d,depth):
    n=len(d); k=int(np.ceil(depth*n)); chosen=d.nlargest(k,"score"); return {"test_n":n,"positives":int(d["y_true"].sum()),"called":k,"captured":int(chosen["y_true"].sum())}
rows=[]
for (model,pid,plabel),d in forward.groupby(["model","period_id","period_label"]):
    for depth in DEPTHS: rows.append({"model":model,"period_id":pid,"period_label":plabel,"call_depth":depth,**period_depth_stats(d,depth)})
period_lift=pd.DataFrame(rows)
def aggregate_lift(d):
    test_rows=int(d.test_n.sum()); positives=int(d.positives.sum()); called=int(d.called.sum()); captured=int(d.captured.sum()); base=positives/test_rows; expected=called*base
    return {"test_rows":test_rows,"test_positives":positives,"base_rate":base,"called":called,"captured":captured,"expected_random_same_volume":expected,"incremental":captured-expected,"lift":(captured/called)/base,"capture_rate":captured/positives}
rng=np.random.default_rng(SEED); lift_rows=[]
for (model,depth),d in period_lift.groupby(["model","call_depth"]):
    a=aggregate_lift(d); arr=d[["test_n","positives","called","captured"]].to_numpy(dtype=float); picks=rng.integers(0,len(arr),size=(2000,len(arr))); sampled=arr[picks].sum(axis=1); test_n_b,pos_b,called_b,captured_b=sampled.T; lift_b=(captured_b/called_b)/(pos_b/test_n_b); a["ci_lower"],a["ci_upper"]=np.quantile(lift_b,[0.025,0.975]); lift_rows.append({"model":model,"call_depth":depth,**a})
lift=pd.DataFrame(lift_rows).sort_values(["model","call_depth"]).reset_index(drop=True); display(lift.round(6))

### 2.1 Lift visualisation

The table above is the numerical source of truth. The chart below visualises how enrichment changes as progressively more of each eligible period is called. A lift of 1.0 represents random calling at the same call volume.

**Coding step.** Compute period-local lift, captured subscribers and random expectation at 5%, 10%, 20% and 50% call depth.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
for model,d in lift.groupby("model"): ax.plot(100*d["call_depth"],d["lift"],marker="o",label=model); ax.fill_between(100*d["call_depth"],d["ci_lower"],d["ci_upper"],alpha=.15)
ax.axhline(1.0,linestyle="--",linewidth=1); ax.set_xlabel("Call depth (% of each eligible period)"); ax.set_ylabel("Lift vs random calling"); ax.set_title("Period-Local Lift by Call Depth"); ax.legend(frameon=False); ax.grid(axis="y",alpha=.25); fig.tight_layout(); plt.show()

## 3. Period-wise calibration

Calibration is assessed within each eligible forward block using score-quantile bins. Brier score and expected calibration error (ECE) are computed per period and aggregated with test-row weights. These are **raw ranking scores**: no sigmoid or isotonic recalibration model is fitted here.

**Coding step.** Calculate period-wise calibration bins, Brier score and expected calibration error without fitting a recalibration model.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
def calibration_bins(d,max_bins=10):
    z=d.copy(); z["bin"]=pd.qcut(z["score"],q=max_bins,duplicates="drop"); return z.groupby("bin",observed=True).agg(n=("y_true","size"),mean_score=("score","mean"),observed_rate=("y_true","mean")).reset_index(drop=True)
cal_rows=[]; curve_parts=[]
for (model,pid,plabel),d in forward.groupby(["model","period_id","period_label"]):
    bins=calibration_bins(d); ece=float(np.sum((bins["n"]/bins["n"].sum())*np.abs(bins["mean_score"]-bins["observed_rate"]))); brier=float(brier_score_loss(d["y_true"],d["score"])); cal_rows.append({"model":model,"period_id":pid,"period_label":plabel,"test_n":len(d),"base_rate":d["y_true"].mean(),"mean_score":d["score"].mean(),"brier":brier,"ece":ece,"roc_auc":roc_auc_score(d["y_true"],d["score"]),"pr_auc":average_precision_score(d["y_true"],d["score"])}); bins=bins.copy(); bins["model"]=model; bins["period_id"]=pid; bins["period_label"]=plabel; curve_parts.append(bins)
calibration_periods=pd.DataFrame(cal_rows); calibration_curve=pd.concat(curve_parts,ignore_index=True); agg_rows=[]
for model,d in calibration_periods.groupby("model"):
    w=d["test_n"]/d["test_n"].sum(); agg_rows.append({"model":model,"periods":len(d),"test_rows":int(d.test_n.sum()),"weighted_base_rate":float(np.sum(w*d.base_rate)),"weighted_mean_score":float(np.sum(w*d.mean_score)),"weighted_brier":float(np.sum(w*d.brier)),"weighted_ece":float(np.sum(w*d.ece)),"weighted_roc_auc":float(np.sum(w*d.roc_auc)),"weighted_pr_auc":float(np.sum(w*d.pr_auc))})
calibration_summary=pd.DataFrame(agg_rows).sort_values("model").reset_index(drop=True); display(calibration_periods.round(6)); display(calibration_summary.round(6))

### 3.1 Calibration visualisation

The calibration plot shows period-local score bins against observed subscription rates. The diagonal is perfect calibration. This is a diagnostic view only: no recalibration model is fitted in this workflow.

**Coding step.** Visualise how historical ranking lift changes with call depth, including bootstrap uncertainty already calculated in the notebook.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
for model,d in calibration_curve.groupby("model"): ax.scatter(d["mean_score"],d["observed_rate"],s=np.clip(d["n"].to_numpy()/8,15,120),alpha=.45,label=model)
ax.plot([0,1],[0,1],linestyle="--",linewidth=1); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xlabel("Mean predicted score"); ax.set_ylabel("Observed subscription rate"); ax.set_title("Period-Wise Calibration Bins"); ax.legend(frameon=False); ax.grid(alpha=.2); fig.tight_layout(); plt.show()

## 4. Final ordered ranking summary

The business-relevant interpretation combines ranking discrimination, call-depth lift and calibration limitations. Model scores should be used for **prioritisation**, not as exact subscription probabilities.

**Coding step.** Combine ordered discrimination, calibration diagnostics and lift into the final ranking summary used for reporting.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
ordered_summary=calibration_summary[["model","periods","test_rows","weighted_roc_auc","weighted_pr_auc","weighted_brier","weighted_ece"]].rename(columns={"weighted_roc_auc":"roc_auc_weighted","weighted_pr_auc":"pr_auc_weighted","weighted_brier":"brier_weighted","weighted_ece":"ece_weighted"}); display(ordered_summary.round(6)); display(lift[lift.call_depth.isin([.05,.10,.20])].round(6))
best_5=lift[lift.call_depth.eq(.05)].sort_values("lift",ascending=False).iloc[0]; print(f"Highest 5% period-local lift: {best_5.model} = {best_5.lift:.4f}; {int(best_5.captured)} subscribers captured from {int(best_5.called)} calls vs {best_5.expected_random_same_volume:.1f} expected under random calling.")

## 5. Notebook-05 conclusion

The forward-period results are the primary robustness evidence. Lift is computed locally within each eligible period, calibration is evaluated period by period, no prospective probability recalibration has been fitted, scores remain ranking signals, and `duration` stays excluded.